# Orchestration Frameworks Overview: Choosing the Right Tool

This notebook is a **comparative reference**, not a tutorial — it does not teach any single
framework from scratch. Instead, it looks across every agent-orchestration framework this repo
already has a built track for and answers three practical questions:

1. What is each framework's **core mental model**?
2. **When** should you reach for it over the alternatives?
3. What does the **same simple task** look like, side by side, in each framework's own idiom?

It closes with a decision-guide table to help you pick a framework for a *new* project, rather
than re-litigating the choice from scratch every time.

**Frameworks covered** (all have a built track elsewhere in this repo — see `NOTEBOOK_INDEX.md`):

| Framework | Where it lives in this repo |
| --- | --- |
| **LangGraph** | `03_LangGraph_Fundamentals/`, `05_AI_Agent_Fundamentals/AI_Agents_with_LangGraph/`, `07_Advanced_Agentic_Systems/` |
| **LangChain** | `02_LangChain_Fundamentals_and_Prompting/`, `05_AI_Agent_Fundamentals/LangChain_Tools_and_Agents/` |
| **CrewAI** | `10_Alternative_Agent_Frameworks/CrewAI/` |
| **AutoGen** | `10_Alternative_Agent_Frameworks/AutoGen/` |
| **DSPy** | `10_Alternative_Agent_Frameworks/DSPy/context-engineering-dspy/` |
| **PydanticAI** | `10_Alternative_Agent_Frameworks/PydanticAI/` — a foundations notebook was just added there in a parallel task; discussed here for completeness but left out of the runnable side-by-side comparison below since its track is brand new |


## 1. Core Abstraction & Mental Model

Each framework encodes a different assumption about what an "agent system" fundamentally *is*.
Knowing this mental model is the fastest way to predict whether a framework will feel natural
or awkward for a given problem.

### LangGraph — explicit state graph
A LangGraph app is a **graph of nodes that read and write a shared, typed state object**. Control
flow (branching, loops, retries, human-in-the-loop interrupts) is drawn explicitly as edges between
nodes rather than inferred from an LLM's free-form conversation. This repo's `03_LangGraph_Fundamentals/`
and `05_AI_Agent_Fundamentals/AI_Agents_with_LangGraph/` tracks build directly on this model.

### LangChain — composable chains (LCEL)
LangChain's mental model is **composable pipelines**: prompts, models, parsers, retrievers and tools
are `Runnable`s that compose with `|` (LCEL) into a chain. It's less about explicit state and more
about linear (or lightly branching) data transformation pipelines — the fundamentals track in
`02_LangChain_Fundamentals_and_Prompting/` and the tool-calling agents in
`05_AI_Agent_Fundamentals/LangChain_Tools_and_Agents/` both build on this.

### CrewAI — role-based crew of agents with tasks
CrewAI models a system as a **crew**: a small team of `Agent`s, each with a `role`, `goal`, and
`backstory`, assigned to `Task`s that a `Crew` executes (sequentially or with `planning=True` for
CrewAI to sequence them itself). It is deliberately closer to "assembling a team" than "wiring a
graph" — see `10_Alternative_Agent_Frameworks/CrewAI/01_Foundations/Some_Simple_Agents/`.

### AutoGen — conversable agents exchanging messages
AutoGen's unit of computation is the **conversable agent**: an `AssistantAgent` or `UserProxyAgent`
that sends and receives chat messages. Multi-agent behavior emerges from agents talking to each
other (`initiate_chat`, group chats, sequential chats) rather than from an explicit graph or crew
object — see `10_Alternative_Agent_Frameworks/AutoGen/`.

### DSPy — declarative signatures + optimizers
DSPy treats prompting as a **compilation problem**. You declare a `Signature` (typed input/output
fields) and a `Module` (e.g. `Predict`, `ChainOfThought`) describing *what* the LLM step should do,
then let a DSPy *optimizer* search for the prompt/few-shot examples that make it work well — rather
than hand-writing and hand-tuning prompt strings, as in `10_Alternative_Agent_Frameworks/DSPy/context-engineering-dspy/`.

### PydanticAI — type-safe agent with Pydantic-validated I/O
PydanticAI's model is an `Agent` whose **output type is a Pydantic model**: the framework validates
(and retries on) the LLM's structured output against that schema, giving strict guarantees that
downstream code can rely on — see `10_Alternative_Agent_Frameworks/PydanticAI/`.


## 2. When to Reach for Each Framework

- **LangGraph**
  - Complex branching or cyclical control flow (loops, retries, conditional routing) that needs fine-grained, inspectable state.
  - Long-running or resumable workflows — human-in-the-loop interrupts, checkpointing, time travel.
  - You want the control flow itself to be reviewable code, not implicit in an LLM's conversation.

- **LangChain**
  - Straightforward, mostly-linear pipelines: prompt to model to parser, or retrieve to augment to generate.
  - You want the largest surface of pre-built integrations (retrievers, document loaders, output parsers) with minimal custom orchestration.
  - A single-agent tool-calling loop is enough — you don't need explicit multi-node state.

- **CrewAI**
  - Quickly standing up a role-based team (e.g. researcher + writer + reviewer) with minimal boilerplate.
  - The problem naturally decomposes into a handful of specialized roles handing off well-defined tasks.
  - You want built-in task planning/sequencing without writing your own graph or message-passing logic.

- **AutoGen**
  - Conversational multi-agent patterns — debate, critique/revise loops, or agents that need to negotiate over several turns.
  - Tasks that benefit from a user-proxy agent that can execute code and relay results back into the conversation.
  - You want the flexibility of raw chat-driven coordination over a more rigid crew/graph structure.

- **DSPy**
  - Prompt-optimization-heavy pipelines where you'd otherwise hand-tune prompts by trial and error.
  - You have (or can build) a metric to optimize against, and want DSPy's compiler to search for better prompts/few-shot examples automatically.
  - Multi-stage LLM pipelines where each stage's prompt should be tuned jointly rather than in isolation.

- **PydanticAI**
  - A system needs **strict output-schema guarantees** with automatic validation/retry — e.g. an agent whose output must always deserialize into a specific Pydantic model.
  - Lightweight, type-first alternative when you don't need LangGraph-style explicit state or CrewAI-style role orchestration.
  - You want typed dependency injection for tools (DB clients, config) without global variables.


## 3. Side-by-Side: The Same Task, Three Frameworks

**Task:** given a topic, research 2-3 facts about it and write a one-paragraph summary.

This is implemented below in **LangGraph**, **CrewAI**, and **AutoGen** — the three frameworks in
this repo with the most directly comparable "multi-step agentic task" tracks. Each snippet uses
that framework's **own native model-configuration convention**, matching how it's already done
elsewhere in this repo:

- LangGraph -> `from helpers import get_llm` (this repo's platform-aware LLM factory).
- CrewAI -> `crewai.LLM(...)`, matching `10_Alternative_Agent_Frameworks/CrewAI/01_Foundations/Some_Simple_Agents/1. Fitness_Progress_Tracker.ipynb`.
- AutoGen -> a `config_list` built from `GROQ_API_KEY`, matching `10_Alternative_Agent_Frameworks/AutoGen/01_Foundations/Some_Simple_Agents/1. Building Agents with AutoGen.ipynb`.

The cells are written to be runnable in principle (given the right API keys/packages installed);
they are left unexecuted here since the point is to compare *shape*, not to produce fresh output.


### 3a. LangGraph — explicit two-node state graph

Two nodes (`research`, `write`) mutate a shared, typed `ResearchState` as the graph walks a fixed
edge from `START` to `research` to `write` to `END`. The state object is the single source of truth
passed between steps.

In [ ]:
# ================================================================================
# LangGraph: research -> write, coordinated through an explicit state graph
# ================================================================================
from typing import TypedDict

from langgraph.graph import END, START, StateGraph

from helpers import get_llm

llm = get_llm()  # platform-aware: Groq on Windows, Databricks on macOS


class ResearchState(TypedDict):
    topic: str
    facts: list[str]
    summary: str


def research_node(state: ResearchState) -> dict:
    prompt = (
        f"List exactly 3 concise, factual bullet points about: {state['topic']}. "
        "Return only the bullets, one per line, no numbering."
    )
    response = llm.invoke(prompt)
    facts = [line.strip("-* ").strip() for line in response.content.splitlines() if line.strip()]
    return {"facts": facts}


def write_node(state: ResearchState) -> dict:
    facts_block = "\n".join(f"- {fact}" for fact in state["facts"])
    prompt = (
        f"Using these facts about {state['topic']}:\n{facts_block}\n\n"
        "Write a single, well-structured paragraph summarizing them for a general audience."
    )
    response = llm.invoke(prompt)
    return {"summary": response.content}


graph = StateGraph(ResearchState)
graph.add_node("research", research_node)
graph.add_node("write", write_node)
graph.add_edge(START, "research")
graph.add_edge("research", "write")
graph.add_edge("write", END)
app = graph.compile()

result = app.invoke({"topic": "the James Webb Space Telescope", "facts": [], "summary": ""})
print(result["summary"])

### 3b. CrewAI — a two-agent crew with sequential tasks

A `Researcher` agent and a `Writer` agent are each given a `Task`; the writer's task declares
`context=[research_task]` so CrewAI passes the research output forward. The `Crew` coordinates
execution — no explicit graph or message loop to write.

In [ ]:
# ================================================================================
# CrewAI: Researcher + Writer agents, coordinated by a Crew
# ================================================================================
from crewai import Agent, Crew, LLM, Task
from dotenv import load_dotenv

load_dotenv()

llm = LLM(model="gpt-4o-mini")

researcher = Agent(
    role="Research Analyst",
    goal="Find 2-3 concise, accurate facts about a given topic",
    backstory="An analyst skilled at quickly surfacing the most relevant facts on any subject.",
    llm=llm,
)

writer = Agent(
    role="Summary Writer",
    goal="Turn a short list of facts into one clear paragraph for a general audience",
    backstory="A writer who specializes in distilling research into a single readable paragraph.",
    llm=llm,
)

topic = "the James Webb Space Telescope"

research_task = Task(
    description=f"Research {topic} and list exactly 3 concise, factual bullet points.",
    expected_output="3 bullet points, one fact per line, no extra commentary.",
    agent=researcher,
)

write_task = Task(
    description=(
        "Using the facts produced by the research task, write a single well-structured "
        "paragraph summarizing them for a general audience."
    ),
    expected_output="One paragraph, no bullet points.",
    agent=writer,
    context=[research_task],
)

crew = Crew(agents=[researcher, writer], tasks=[research_task, write_task])
result = crew.kickoff()
print(result.raw)

### 3c. AutoGen — two assistant agents driven through a user proxy

There is no shared state object and no crew object — a `UserProxyAgent` relays messages between
two `AssistantAgent`s (`researcher`, `writer`) one turn at a time. Coordination is just who said
what to whom.

In [ ]:
# ================================================================================
# AutoGen: researcher and writer AssistantAgents, relayed via a UserProxyAgent
# ================================================================================
import os

import autogen
from autogen import AssistantAgent, UserProxyAgent
from dotenv import load_dotenv

load_dotenv()

api_key = os.environ.get("GROQ_API_KEY")
config_list = [{
    "model": "llama-3.3-70b-versatile",
    "api_key": api_key,
    "api_type": "groq",
}]

researcher = AssistantAgent(
    name="researcher",
    system_message=(
        "You research topics and reply with exactly 3 concise, factual bullet points. "
        "Never write anything other than the bullets."
    ),
    llm_config={"config_list": config_list},
)

writer = AssistantAgent(
    name="writer",
    system_message=(
        "You take bullet-point facts from the previous message and rewrite them as a single "
        "well-structured paragraph for a general audience. Reply with ONLY the paragraph, "
        "then say TERMINATE."
    ),
    llm_config={"config_list": config_list},
)

user_proxy = UserProxyAgent(
    name="user_proxy",
    code_execution_config=False,
    human_input_mode="NEVER",
    max_consecutive_auto_reply=1,
    is_termination_msg=lambda msg: "TERMINATE" in msg.get("content", ""),
)

topic = "the James Webb Space Telescope"

# Turn 1: researcher produces the facts
user_proxy.initiate_chat(researcher, message=f"Research this topic: {topic}", max_turns=1)
facts = user_proxy.last_message(researcher)["content"]

# Turn 2: writer turns those facts into a paragraph
user_proxy.initiate_chat(writer, message=facts, max_turns=1)
summary = user_proxy.last_message(writer)["content"]

print(summary)

**What the comparison shows:** all three snippets do the same two-step "research then write"
job, but the *unit of coordination* differs — a state dict passed along explicit edges (LangGraph),
a task's declared `context` dependency inside a crew (CrewAI), or a literal chat message relayed by
a proxy (AutoGen). None is strictly more code than another for this simple case; the difference
shows up as task complexity grows (see the decision guide below).

## 4. Decision Guide

| Framework | Learning curve | Control granularity | Best-fit team/task size | Ecosystem maturity in this repo |
| --- | --- | --- | --- | --- |
| **LangGraph** | Moderate-steep (explicit graph/state mental model) | Very high — every transition, retry, and interrupt is explicit code | Small to large; scales well to complex multi-step or cyclical workflows | High — Phases 3, 5, 7 all build on it extensively |
| **LangChain** | Low-moderate (LCEL piping) | Moderate — great for linear/branching chains, less for cyclical control | Small; single-agent or simple tool-calling agents | High — Phase 2 fundamentals + Phase 5 tool-calling agents |
| **CrewAI** | Low (role/task/crew vocabulary is intuitive) | Low-moderate — coordination is mostly delegated to the Crew | Small teams of 2-5 role-based agents | Moderate-high — Phase 10 has foundations, flows, multi-agent patterns, 9 applied projects |
| **AutoGen** | Moderate (conversation/config-list concepts) | Moderate — explicit per-turn control via `initiate_chat`/group chats | Small to medium; strong for conversational back-and-forth patterns | Moderate-high — Phase 10 has foundations, core capability labs, group/swarm patterns, 8 project sets |
| **DSPy** | Steep (signatures, modules, optimizers is a different paradigm) | High, but over *prompts* rather than control flow | Any size, but pays off most on pipelines with a clear optimization metric | Narrow but deep — one focused, multi-level track (`context-engineering-dspy`) |
| **PydanticAI** | Low (Pydantic-familiar API) | High on output validation, lighter on broader multi-agent orchestration | Fits small agents needing strict schema guarantees | New — a foundations track was just added in `10_Alternative_Agent_Frameworks/PydanticAI/` |

**Rule of thumb:** reach for **LangGraph** when control flow itself is the hard part; **LangChain**
when it's a simple pipeline with rich integrations; **CrewAI** when the natural decomposition is a
small team of roles; **AutoGen** when the natural decomposition is a conversation between agents;
and **DSPy** when the hard part is finding the right prompt, not the right control flow.


## Summary & Key Takeaways

- **Mental models differ more than capability.** All of these frameworks can technically build a
  multi-step agentic pipeline; what differs is which mental model — graph, chain, crew,
  conversation, or optimizer — makes *your specific* problem easiest to reason about and maintain.
- **LangGraph** earns its place in this repo for anything needing explicit, inspectable control
  flow (branching, cycles, human-in-the-loop, checkpointing).
- **LangChain** remains the fastest path for simple, mostly-linear pipelines with rich pre-built
  integrations.
- **CrewAI** and **AutoGen** both target multi-agent collaboration but from different angles —
  CrewAI via role/task delegation to a Crew, AutoGen via direct agent-to-agent conversation. Pick
  based on whether "team with tasks" or "conversation between agents" better matches your problem.
- **DSPy** is the outlier: it's not about orchestrating *agents* at all, but about compiling better
  *prompts* for the LLM calls inside a pipeline — reach for it when prompt quality, not control
  flow, is the bottleneck.
- **PydanticAI** is the pick when strict, validated-output agents matter more than complex
  orchestration — see its own new foundations notebook in this phase for a deeper walkthrough.
- Use the decision-guide table above as a starting filter, then confirm the pick against the
  concrete side-by-side code in Section 3 — seeing the same task solved three ways is usually more
  convincing than any table.
